# HR 子模块数据合成 Notebook

本 Notebook 参考 ToolOrchestra 的设计，为 HR 系统的每个子模块自动生成：
- Data Model（Pydantic 模型）
- Data Schema（JSON Schema）
- Database Content（测试数据）
- Tools（工具函数类）
- Tasks（简单和复杂任务）
- Ground Truth（验证代码）

## Cell 1: 初始化

In [ ]:
import json
from pathlib import Path
from synthesis.llm_client import LLMClient
from synthesis.data_utils import save_generated_file, load_json, extract_code_block, verify_json_structure
from loguru import logger

# 初始化 LLM 客户端
llm = LLMClient("claude-sonnet-4-6")

# 配置路径
PROMPTS_DIR = Path("prompts")
GENERATED_DIR = Path("generated")
BASE_DIR = GENERATED_DIR / "base"

print("✓ 初始化完成")
DOMAIN = "it"
MODULES_DIR = GENERATED_DIR / f"{DOMAIN}"

## Cell 3: 生成 Schema（JSON Schema）

In [ ]:
# 加载 base schema
import time
base_schema = load_json(BASE_DIR / "schema.json")

# 加载 schema prompt 模板
with open(PROMPTS_DIR / f"{DOMAIN}" / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", json.dumps(base_schema, indent=2, ensure_ascii=False))

# 调用 LLM 生成 schema
# print(f"正在生成 {DOMAIN} 的 Schema, prompt: \n{schema_prompt}")
# schema_response = llm.generate(schema_prompt)
# print(schema_response.content)
print("开始生成数据...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    schema_response += content
    print(content, end="", flush=True)
usage = getattr(chunk, "usage_metadata", None)
print(f"\n{'-' * 60}")
schema_text = extract_code_block(schema_response)
print(schema_text)
print(usage)
# 保存
schema_path = MODULES_DIR / DOMAIN / "schema.json"
save_generated_file(schema_path, schema_text)
print(f"✓ Schema 已保存到: {schema_path}")

In [ ]:
# 加载 base models
with open(BASE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()

with open(MODULES_DIR / DOMAIN / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

# 加载 data model prompt 模板
with open(PROMPTS_DIR / f"{DOMAIN}" / "2_data_models.txt", 'r', encoding='utf-8') as f:
    model_prompt = f.read()

model_prompt = model_prompt.replace("{base_models}", base_models)
model_prompt = model_prompt.replace("{domain_schema}", domain_schema)

# print(model_prompt)
# 调用 LLM 生成 data model
print(f"正在生成 {DOMAIN} 的 Data Model...")
print("-" * 60)
model_response = ""
for chunk in llm.llm.stream(model_prompt):
    content = chunk.content
    model_response += content
    print(content, end="", flush=True)
usage = getattr(chunk, "usage_metadata", None)
print(f"\n{'-' * 60}")
model_code = extract_code_block(model_response)
print(model_code)
print(usage)
# 保存
model_path = MODULES_DIR / DOMAIN / "models.py"
save_generated_file(model_path, model_code)

print(f"✓ Data Model 已保存到: {model_path}")

In [ ]:
def load_schema_and_model():
    with open(MODULES_DIR / DOMAIN / "schema.json", 'r', encoding='utf-8') as f:
        module_schema = f.read()

    with open(MODULES_DIR / DOMAIN / "models.py", 'r', encoding='utf-8') as f:
        model_code = f.read()

    return module_schema, model_code

In [ ]:
module_schema, model_code = load_schema_and_model()


validation_prompt = f"""
Schema:
{module_schema}

Data model:
{model_code}

请检查以下两点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致（例如，同一变量在不同地方使用相同的类型）

如果是no，给出修改意见
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = llm.generate(validation_prompt)

print(f"\n验证结果：")
print(validation_response)

In [ ]:
module_schema, model_code = load_schema_and_model()

regenerate_prompt = f"""当前的schema为：{module_schema}
当前需要生成的model为：{model_code}
校验结果为：{validation_response}
请根据校验结果修改model
"""
response = llm.generate(regenerate_prompt)
model_code = extract_code_block(response)

# 保存
model_path = MODULES_DIR / DOMAIN / "models.py"
save_generated_file(model_path, model_code)

print(f"✓ Data Model 已保存到: {model_path}")  

In [ ]:

module_schema, model_code = load_schema_and_model()
with open(BASE_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data = f.read()
with open(PROMPTS_DIR / f"{DOMAIN}" / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    db_prompt = f.read()

db_prompt = db_prompt.replace("{base_data}", base_data)
db_prompt = db_prompt.replace("{model_code}", model_code)

print(f"{'-' * 60}")
print(f"正在生成 {DOMAIN} 的测试数据生成脚本...")
db_response = ""

for chunk in llm.llm.stream(db_prompt):
    content = chunk.content
    db_response += content
    print(content, end="", flush=True)
usage = getattr(chunk, "usage_metadata", None)
print(f"\n{'-' * 60}")
print(usage)

# db_content = extract_code_block(db_response)
# is_json, db_data = verify_json_structure(db_content)
# if db_data:
#     print("✓ JSON 结构验证通过")
# else:
#     print("✗ JSON 结构验证失败，请检查生成的数据")

# 保存
# db_path = MODULES_DIR / DOMAIN / "db0.json"
# save_generated_file(db_path, db_content)
# print(f"✓ Database Content 已保存到: {db_path}")

In [ ]:

module_schema, model_code = load_schema_and_model()
with open(BASE_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data = f.read()
with open(PROMPTS_DIR / f"{DOMAIN}" / "3_database_content.txt", 'r', encoding='utf-8') as f:
    db_prompt = f.read()

db_prompt = db_prompt.replace("{base_data}", base_data)
db_prompt = db_prompt.replace("{model_code}", model_code)

print(f"{'-' * 60}")
print(f"正在生成 {DOMAIN} 的测试数据...")
db_response = ""

for chunk in llm.llm.stream(db_prompt):
    content = chunk.content
    db_response += content
    print(content, end="", flush=True)
usage = getattr(chunk, "usage_metadata", None)
print(f"\n{'-' * 60}")
print(usage)
db_content = extract_code_block(db_response)
is_json, db_data = verify_json_structure(db_content)
if db_data:
    print("✓ JSON 结构验证通过")
else:
    print("✗ JSON 结构验证失败，请检查生成的数据")

# 保存
db_path = MODULES_DIR / DOMAIN / "db0.json"
save_generated_file(db_path, db_content)
print(f"✓ Database Content 已保存到: {db_path}")

In [ ]:
import json
from generated.domains.IT.models import ITDB

with open(GENERATED_DIR / "domains" /  "IT" / "db0.json", "r", encoding="utf-8") as f:
    data = json.load(f)

try:
    db  = ITDB.model_validate(data)
    print("成功转换！")
    print(db.get_statistics())

except Exception as e:
    print(f"验证失败: {e}")

In [ ]:
with open(MODULES_DIR / DOMAIN / "db.json", "r", encoding="utf-8") as f:
    db_content = f.read()
db_check_prompt = f'''{db_content}

Please check whether the content is natural and reasonable.
Please also check whether content is consistent, e.g., client id is the same across different places if it refers to the same client.
Start your answer with yes if all requirements are satisfied, and start your answer with no otherwise.'''
response = llm.generate(db_check_prompt)
print(response)

## Cell 7: 生成 Tools（工具函数）

In [ ]:
function_description = """### 薪酬与福利 (Compensation & Benefits)
工资、奖金、津贴、社保公积金、商业保险、异地福利

Functions: 基本工资查询、奖金咨询、津贴补贴、社保管理、公积金管理、商业保险

### 工作安排与假期 (Work Schedule & Leave)
法定节假日、福利假、请假、加班

Functions: 法定节假日咨询、公司福利假咨询、请假管理、加班管理

### 入职与离职管理 (Onboarding & Offboarding)
入职、离职、岗位调整、晋升等

Functions: 入职管理、离职管理、转岗管理、晋升管理

### 招聘与引进 (Recruitment)
校招、社招、内推、外包管理、实习生

Functions: 校招、社招、内推、外包入职、实习生招聘、面试、入职

### 培训与发展 (Training & Development)
新人培训、职业发展、试用期考核、导师制度

Functions: 新人培训、试用期考核、导师/师兄制度

### 绩效与激励 (Performance & Incentives)
绩效考核、股权激励、人才补贴、职称管理

Functions: 绩效管理、股权激励、长期激励

### 档案与证明 (Records & Certificates)
人事档案、户口管理、居住证明、党团关系、签证管理

Functions: 人事档案、居住证明、签证管理

### 税务与合规 (Tax & Compliance)
个税申报、个税测算、违规举报

Functions: 个税申报、违规举报

### HR服务 (HR Services)
热线、服务台、证明开具等

Functions: HR热线查询、HR服务台查询、证明开具
"""


module_schema, model_code = load_schema_and_model()

# 加载 tools prompt 模板
with open(PROMPTS_DIR / "it" / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", function_description)

# 调用 LLM 生成工具函数
print(f"{'-' * 60}")
tools_response = ""

for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)
usage = getattr(chunk, "usage_metadata", None)
print(f"\n{'-' * 60}")
print(usage)
print(f"正在生成 {DOMAIN} 的 Tools...")
tools_code = extract_code_block(tools_response)

# 保存
tools_path = MODULES_DIR / DOMAIN / "tools.py"
save_generated_file(tools_path, tools_code)

print(f"✓ Tools 已保存到: {tools_path}")

# 验证 Python 语法
from synthesis.data_utils import verify_python_syntax
error = verify_python_syntax(tools_code)
if not error[0]:
    print(f"\n✗ Python 语法错误: {error}")
else:
    print(f"✓ Python 语法验证通过")
    print(f"\n生成的 Tools 预览：")

In [ ]:
subject_prompt = f'''IT域当前的现有工具: 
{tools_code}
请你帮面整理和提出在{DOMAIN}中提出100个日常生活中常见的真实目的。
我偏好需要多个步骤才能解决的复杂目的。
每个目的应该只用几个词表达。
使用以下格式输出：
[
    "目的 1",
    "目的 2"
    ...
]
'''
# print(subject_prompt)
purpose = llm.generate(subject_prompt)
print(purpose)

In [ ]:
purposes = [
    "查询本月工资明细",
    "申请年假并提交审批",
    "查询剩余年假天数",
    "申请加班并计算调休",
    "办理入职手续",
    "查询社保公积金缴纳情况",
    "申请商业保险报销",
    "查询法定节假日安排",
    "提交离职申请",
    "办理工作地点调动",
    "申请部门转岗",
    "查询绩效考核结果",
    "申请开具在职证明",
    "办理居住证明",
    "查询个税扣缴明细",
    "进行年度个税汇算",
    "申请病假并上传证明",
    "查询试用期结束时间",
    "申请晋升评审",
    "查询培训课程安排",
    "报名参加新员工培训",
    "申请导师制度配对",
    "查询股权激励明细",
    "办理劳动合同续签",
    "申请异地福利补贴",
    "查询加班时长统计",
    "申请调整工作班次",
    "办理签证申请手续",
    "提交违规行为举报",
    "查询招聘岗位信息",
    "推荐候选人内推",
    "安排面试时间",
    "查询面试反馈结果",
    "发放录用通知书",
    "查询职称评定进度",
    "申请专业技能认证",
    "办理人事档案调转",
    "查询薪资调整历史",
    "申请事假审批",
    "查询考勤异常记录",
    "申请远程办公",
    "办理产假手续",
    "查询陪产假天数",
    "申请哺乳假时间",
    "办理婚假申请",
    "查询丧假政策",
    "申请工伤认定",
    "查询补充医疗保险",
    "办理退休手续准备",
    "申请提前离职",
    "查询离职补偿标准",
    "办理离职交接",
    "申请调整薪资结构",
    "查询奖金发放时间",
    "申请绩效申诉",
    "查询部门组织架构",
    "申请更换直属主管",
    "办理跨国派遣手续",
    "查询外派补贴政策",
    "申请回国定居",
    "办理工作签证延期",
    "查询劳务派遣合同",
    "申请转正式员工",
    "查询实习生转正条件",
    "办理学生实习协议",
    "申请校园招聘参与",
    "查询社会招聘流程",
    "办理外包人员入场",
    "申请背景调查",
    "查询试用期考核标准",
    "提交试用期自评",
    "申请延长试用期",
    "办理提前转正",
    "查询培训学分要求",
    "申请外部培训报销",
    "办理职业资格认证",
    "查询继续教育补贴",
    "申请学历提升资助",
    "办理技能等级晋升",
    "查询岗位说明书",
    "申请岗位职责调整",
    "办理兼职审批",
    "查询竞业限制条款",
    "申请解除竞业协议",
    "办理保密协议签署",
    "查询知识产权归属",
    "申请专利奖励",
    "办理年度体检预约",
    "查询员工福利清单",
    "申请困难补助",
    "办理住房公积金提取",
    "查询企业年金账户",
    "申请补充养老保险",
    "办理员工关怀慰问",
    "查询工会会员权益",
    "申请HR服务台支持",
    "拨打HR热线咨询",
    "查询人力资源政策",
    "申请薪酬保密协议"
]

purposes = [
    "为新员工分配笔记本电脑并安装办公软件",
    "员工离职时回收所有设备和吊销账号权限",
    "批量检查并修复不合规MDM设备",
    "为跨部门项目组临时授予VPN访问权限",
    "处理员工报告的笔记本电脑丢失并更新资产状态",
    "审批并跟踪IT设备采购订单直至交付",
    "将即将到期的软件合同续签前评估供应商评分",
    "为远程员工配置软件许可并绑定设备",
    "调查高风险安全日志并创建处理工单",
    "统计部门预算使用情况并记录新增支出",
    "将维修完成的设备重新分配给等待员工",
    "为新项目申请网络资源并分配IP地址",
    "批量撤销离职员工的所有软件授权",
    "将高优先级故障工单升级并指派给专职工程师",
    "检查软件许可证剩余席位并决定是否采购",
    "为实习生配置临时账号和有限期访问权限",
    "定期审查并清理长期未使用的访问授权",
    "将老旧设备标记退役并记录退役原因",
    "响应员工密码重置请求并启用MFA",
    "追踪采购订单状态并催促供应商交货",
    "为安全审计整理员工账号MFA启用情况",
    "将软件许可从离职员工转移给新员工",
    "处理员工反映的网络连接故障工单",
    "评估并关闭已解决的IT支持工单",
    "为高管配置专属设备并优先安排软件安装",
    "核查MDM合规率并生成不合规设备清单",
    "为季度IT预算制定支出计划并跟踪执行",
    "将测试环境设备转移到生产部门使用",
    "处理员工账号被锁定的紧急恢复请求",
    "审查即将到期的供应商合同并安排续签",
    "为新办公室配置网络设备并分配IP段",
    "将软件安装工单关联到知识库解决方案",
    "批量更新设备操作系统版本并验证合规",
    "为临时外包人员开通有限期系统访问",
    "统计各部门资产分布并优化资源配置",
    "处理设备硬件故障并安排维修或更换",
    "为员工出差配置临时VPN和远程访问权限",
    "审核并批准IT部门月度采购申请",
    "追踪关键合同到期日期并提前预警",
    "为新业务系统上线分配软件许可席位",
    "处理员工反映的软件崩溃问题并记录工单",
    "将闲置设备从仓库分配给有需求的部门",
    "为安全事件响应撤销涉事账号的所有权限",
    "检查并更新资产的物理位置信息",
    "为员工提升职级后更新系统访问权限",
    "统计软件许可使用率并决定是否缩减采购",
    "处理员工反映的邮件账号异常登录警告",
    "为IT资产盘点核对账面与实物一致性",
    "将网络故障工单分配给网络工程师并跟踪",
    "为新入职批量员工完成系统账号开通",
    "审查高风险安全事件并决定是否上报",
    "为设备更换场景完成旧设备回收和新设备发放",
    "处理软件许可超量使用警告并调整分配",
    "为跨国团队配置多时区访问策略",
    "将已解决工单关联满意度评分并归档",
    "检查并修复设备MDM签到异常问题",
    "为年度安全审计整理访问权限变更记录",
    "处理员工反映的打印机无法连接问题",
    "为新供应商合同录入并设置到期提醒",
    "统计各类工单处理时效并找出瓶颈",
    "为员工配置双显示器并更新资产记录",
    "处理设备被盗事件并启动应急响应流程",
    "为研发部门申请高配置工作站并跟踪采购",
    "审查并停用长期不活跃的系统账号",
    "为合规检查整理所有临时访问授权记录",
    "将维护中的设备恢复可用并通知等待员工",
    "处理员工反映的VPN连接不稳定问题",
    "为新软件上线前完成许可证采购和分配",
    "统计本季度IT支出并与预算对比分析",
    "为员工配置移动设备MDM并验证合规",
    "处理网络IP地址冲突并释放占用地址",
    "为安全培训后批量启用员工账号MFA",
    "审查并拒绝不合规的采购申请",
    "为设备报废前完成数据清除和资产注销",
    "处理员工反映的账号权限不足问题",
    "为新建项目团队批量开通协作工具许可",
    "统计各部门软件许可使用情况并优化分配",
    "为紧急业务需求快速审批采购订单",
    "处理员工反映的系统登录异常安全事件",
    "为资产全生命周期管理建立完整事件记录",
    "审查并更新供应商合同绩效评分",
    "为员工配置新手机并完成MDM注册",
    "处理部门预算超支预警并调整支出计划",
    "为知识库文章关联常见故障工单",
    "统计并报告MDM设备合规率趋势",
    "为员工账号状态变更完成审批和执行",
    "处理多个员工同时报告的网络中断事件",
    "为年度IT资产审计准备完整资产清单",
    "审查并清理已过期的临时访问授权",
    "为新业务线配置独立预算并跟踪使用",
    "处理软件版本升级引发的兼容性问题工单",
    "为员工工位搬迁更新设备位置和网络配置",
    "统计知识库文章访问热度并优化内容",
    "为安全漏洞响应紧急撤销相关系统访问",
    "处理采购订单交付异常并协调供应商",
    "为员工配置开发环境所需全套软件和权限",
    "审查并归档已完成的IT采购和合同记录",
    "为跨部门资产共享制定借用和归还流程",
    "处理员工反映的账号被盗用紧急安全事件"
]

## Cell 8: 生成 Simple Tasks（简单任务）

In [ ]:
# 加载 simple task prompt 模板

with open(BASE_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_content = f.read()
with open(MODULES_DIR / DOMAIN / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()
with open(MODULES_DIR / DOMAIN / "db0.json", 'r', encoding='utf-8') as f:
    module_content = f.read()
# 填充 prompt
tasks = []
simple_tasks_path = MODULES_DIR / DOMAIN / "tasks.json"
max_try = 3
for purpose in purposes:
    for attempt in range(3):
        try:
            with open(PROMPTS_DIR / "it" / "5_domain_tasks.txt", 'r', encoding='utf-8') as f:
                task_prompt = f.read()
            print(f"正在生成 {DOMAIN} 的 Task: {purpose} (第{attempt+1}次)")
            task_prompt = task_prompt.replace("{tool_code}", tools_code)
            task_prompt = task_prompt.replace("{module_content}", module_content)
            task_prompt = task_prompt.replace("{base_content}", base_content)
            task_prompt = task_prompt.replace("{purpose}", purpose)
            if attempt == 0:
                simple_tasks_response = llm.generate(task_prompt)
            else:
                simple_tasks_response = llm.generate(task_prompt + "\nmake sure it's in corret json format")
            simple_tasks_text = extract_code_block(simple_tasks_response)
            simple_tasks_text = json.loads(simple_tasks_text)
            print(simple_tasks_text)
            tasks.append(simple_tasks_text)

            tasks_json = json.dumps(tasks, indent=2, ensure_ascii=False)
            save_generated_file(simple_tasks_path, tasks_json)
            break  # 成功则跳出重试循环，进入下一个 purpose
        except Exception as e:
            print(f"生成 {purpose} 失败 (第{attempt+1}次): {e}")
            if attempt == 2:
                print(f"跳过 {purpose}")
print(f"✓ Simple Tasks 已保存到: {simple_tasks_path}")

# 验证 JSON 格式
# error = verify_json_structure(simple_tasks_text)
# if error:
#     print(f"\n✗ JSON 格式错误: {error}")
# else:
#     print(f"✓ JSON 格式验证通过")
#     simple_tasks = json.loads(simple_tasks_text)
#     print(f"\n生成了 {len(simple_tasks)} 个简单任务")
#     for i, task in enumerate(simple_tasks[:3], 1):
#         print(f"\n任务 {i}: {task.get('user_scenario', 'N/A')[:80]}...")

## Cell 9: 生成 Complex Tasks（复杂任务）

In [ ]:
with open(PROMPTS_DIR / "module" / "6_evolve_task.txt", 'r', encoding='utf-8') as f:
    evolve_task_prompt = f.read()
with open(BASE_DIR / "0.json", 'r', encoding='utf-8') as f:
    base_content = f.read()
with open(MODULES_DIR / SELECTED_MODULE / "0.json", 'r', encoding='utf-8') as f:
    module_content = f.read()


# 填充 prompt
evolve_task_prompt = evolve_task_prompt.replace("{module_name}", SELECTED_MODULE)
evolve_task_prompt = evolve_task_prompt.replace("{tool_code}", tools_code)
evolve_task_prompt = evolve_task_prompt.replace("{module_content}", module_content)
evolve_task_prompt = evolve_task_prompt.replace("{base_content}", base_content)
evolve_task_prompt = evolve_task_prompt.replace("{raw_task}", simple_tasks_text)


# 调用 LLM 生成复杂任务
# print(evolve_task_prompt)
print(f"正在生成 {SELECTED_MODULE} 的 Complex Tasks...")
complex_tasks_response = llm.generate(evolve_task_prompt)

# # 提取代码块
complex_tasks_text = extract_code_block(complex_tasks_response)
print(complex_tasks_text)

# # 保存
# complex_tasks_path = MODULES_DIR / SELECTED_MODULE / "complex_tasks.json"
# save_generated_file(complex_tasks_path, complex_tasks_text)

# print(f"✓ Complex Tasks 已保存到: {complex_tasks_path}")

# # 验证 JSON 格式
# error = verify_json_structure(complex_tasks_text)
# if error:
#     print(f"\n✗ JSON 格式错误: {error}")
# else:
#     print(f"✓ JSON 格式验证通过")
#     complex_tasks = json.loads(complex_tasks_text)
#     print(f"\n生成了 {len(complex_tasks)} 个复杂任务")
#     for i, task in enumerate(complex_tasks, 1):
#         print(f"\n任务 {i}: {task.get('user_scenario', 'N/A')}")